In [2]:
import os
import pandas as pd
import re



In [5]:
import pandas as pd
import os


def process_trips(file_path):
    # Read the CSV file
    df = pd.read_csv(file_path, low_memory=False)


    # Check if required columns exist
    # required_columns = ['Engine_speed', 'Trip_fuel_consumption', 'Time', 'Battery_voltage', 'Average_fuel_consumption_rate']
    required_columns = ['Engine_speed', 'Trip_fuel_consumption', 'Time', 'Battery_voltage']

    if not all(col in df.columns for col in required_columns):
        print(f"One of the required columns {required_columns} not found in {file_path}. Skipping this file.")
        return None

    # Convert columns to numeric, forcing errors to NaN
    df['Engine_speed'] = pd.to_numeric(df['Engine_speed'], errors='coerce')
    df['Trip_fuel_consumption'] = pd.to_numeric(df['Trip_fuel_consumption'], errors='coerce')
    df['Time'] = pd.to_numeric(df['Time'], errors='coerce')
    df['Battery_voltage'] = pd.to_numeric(df['Battery_voltage'], errors='coerce')
    df['Cumulative_mileage'] = pd.to_numeric(df['Cumulative_mileage'], errors='coerce')

    df['Trip_fuel_consumption'] = df['Trip_fuel_consumption'] / 1000000

    df['Time'] = pd.to_numeric(df['Time'], errors='coerce')
    df['Time'] = df['Time'] / 3600000  # Convert time to hours
    # Initialize variables for trip tracking
    trip_number = 1
    trip_data = []
    current_trip = []

    # Loop through rows to identify trips
    for index, row in df.iterrows():
        engine_speed = row['Engine_speed']
        battery_voltage = row['Battery_voltage']

        if engine_speed > 0 and battery_voltage > 0:  # Start or continue a trip
            current_trip.append(row)
        else:  # Engine speed is zero, end the current trip
            if current_trip:
                trip_data.append(current_trip)
                current_trip = []

    # If there's data left in the current trip, save it
    if current_trip:
        trip_data.append(current_trip)

    # Process each trip and calculate Instant_fuel_consumption_lph
    processed_trips = []
    for idx, trip in enumerate(trip_data):
        trip_df = pd.DataFrame(trip)

        # Calculate Instant_fuel_consumption_lph
        trip_df['Instant_fuel_consumption_lph'] = (trip_df['Trip_fuel_consumption'].diff().shift(-1) / trip_df['Time'].diff().shift(-1)) * 3.6

        # Remove rows where Instant_fuel_consumption_lph is <= 0
        trip_df = trip_df[trip_df['Instant_fuel_consumption_lph'] >= 0].reset_index(drop=True)
        trip_df['Trip_Number'] = idx + 1
        # Add the processed trip to the list
        processed_trips.append(trip_df)

        # Save the trip DataFrame to a new CSV file in the output folder
        if not trip_df.empty:
            
            trip_number += 1

    # Combine all processed trips back into a single DataFrame
    final_df = pd.concat(processed_trips, ignore_index=True)

    return final_df


In [6]:
# Define threshold for filtering folders
THRESHOLD = 0  # Replace with your X value
THRESHOLD2 = 10
output_file = 'output.xlsx'

# Initialize an empty DataFrame to collect data
final_data = []

# Get list of folders in the directory
base_dir = r'\\fileserver3\Inventory\1- Product Development\10- IPCO data Lake\1-Data Loggers'  # Replace with your directory path


folders = [f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f))]

for folder in folders:
    try:
        # Check if folder starts with 4 digits
        if re.match(r'^\d{4}', folder):
            parts = re.split(r'[_\(\)]', folder)
            
            if len(parts) >= 4:
                range_value = parts[0]  # Example: `0118-0212
                range_part = parts[4]  # Example: `37-638`
                # print(range_part)
                number_part = range_value.split('-')[0]  # Example: `0118`
                # print(number_part)
                # Check if number part meets the condition
                # if int(number_part) > THRESHOLD and int(number_part) < THRESHOLD2:
                if int(number_part) > THRESHOLD:

                    folder_path = os.path.join(base_dir, folder)

                    # Locate the required files
                    info_file = None
                    csv_file = None
                    for file in os.listdir(folder_path):
                        if file.startswith('info_') and file.endswith('.dat'):
                            info_file = os.path.join(folder_path, file)
                        elif file == 'One_BIGfile_Refined.csv':
                            csv_file = os.path.join(folder_path, file)


                    if info_file :
                        # Read info_xxxxx.dat
                        info_data = {}
                        # Read info_xxxxx.dat
                        

                        try:
                            with open(info_file, 'r', encoding='utf-8') as f:
                                for line in f:
                                    # print(f"Raw line: {line}")  # Debug: Print raw line
                                    line = line.strip()
                                    # print(f"Processed line: {line}")  # Debug: Print stripped line
                                    
                                    if '=' in line:  # Check for valid key-value pairs (now using '=')
                                        try:
                                            # Attempt to split the line
                                            key, value = line.split('=', 1)
                                            key = key.strip()  # Clean up the key
                                            value = value.strip()  # Clean up the value
                                            
                                            # Add the key-value pair to the dictionary
                                            info_data[key] = value
                                            # print(f"Added to dictionary: {key} -> {value}")  # Debug: Print success
                                        except Exception as e:
                                            print(f"Error splitting line: {line} - {e}")
                                    else:
                                        print(f"Skipping invalid line (no equal sign found): {line}")
                        except FileNotFoundError:
                            print(f"File not found: {info_file}")
                        except Exception as e:
                            print(f"Error reading file {info_file}: {e}")

                        # print("Final info_data:", info_data)
                    else :
                        # Read info_xxxxx.dat
                        info_data = {
                        'Name': '',
                        'Vehicle model': '',
                        'Vehicle option': '',
                        'Start odometer': '0',
                        'End odometer': '0',
                        'Installation date': '',
                        'Removal date': ''
                    }
                        
                    if csv_file :


                        # print(info_data)
                        processed_df = process_trips(csv_file)
                        if processed_df is not None:
                            # Group by trip and calculate metrics
                            
                            trip_data = processed_df.groupby('Trip_Number').agg({
                                'Cumulative_mileage': ['sum', lambda x: x.iloc[-1] - x.iloc[0]],  # Difference between last and first
                                'Time': ['sum', lambda x: x.iloc[-1] - x.iloc[0]],                # Same for Time
                                'Trip_fuel_consumption': ['sum', lambda x: x.iloc[-1] - x.iloc[0]]  # Same for Fuel consumption
                            }).reset_index()


                            # Flatten the MultiIndex columns
                            trip_data.columns = ['Trip_Number', 'Mileage_sum', 'Mileage_diff', 'Time_sum', 'Time_diff', 'Fuel_consumption_sum', 'Fuel_consumption_diff']

                            # trip_data['Time_sum'] = trip_data['Time_sum']/3600000

                            # trip_data['Fuel_consumption_sum'] = trip_data['Fuel_consumption_sum']/1000000
                            # print(trip_data['Fuel_consumption_sum'])
                            # print(trip_data['Mileage_sum'])
                            # Calculate average vehicle speed (mileage / time)
                            trip_data['Average_Vehicle_Speed'] = trip_data['Mileage_sum'] / trip_data['Time_sum']

                            # trip_data['fuel_consumption_avg'] = trip_data['Fuel_consumption_sum'] / trip_data['Mileage_sum']

                            # Add folder and info data to each row
                            # trip_data['plate_number'] = range_part
                            # trip_data['Range'] = number_part
                            # trip_data['driver_name'] = info_data.get('Name', '')
                            # trip_data['vehicle_model'] = info_data.get('Vehicle model', '')
                            # trip_data['vehicle_option'] = info_data.get('Vehicle option', '')
                            # trip_data['Start_Odo'] = info_data.get('Start odometer', '')
                            # trip_data['End_Odo'] = info_data.get('End odometer', '')
                            # trip_data['Start_Date'] = info_data.get('Installation date', '')
                            # trip_data['End_Date'] = info_data.get('Removal date', '')


                            start_odo = info_data.get('Start odometer', '').strip()
                            end_odo = info_data.get('End odometer', '').strip()

                            # Convert to integers if they are numeric, otherwise default to 0
                            start_odo = int(start_odo) if start_odo.isdigit() else 0
                            end_odo = int(end_odo) if end_odo.isdigit() else 0


                            # Calculate overall metrics for all trips
                            overall_metrics = {
                                'plate_number': range_part,
                                'range': number_part,
                                'driver_name': info_data.get('Name', ''),
                                'vehicle_model': info_data.get('Vehicle model', ''),
                                'vehicle_option': info_data.get('Vehicle option', ''),
                                'engine-type' : '',
                                'transmission-type' : '',
                                'start_odo': info_data.get('Start odometer', ''),
                                'end_odo': info_data.get('End odometer', ''),
                                'mileage_odo' : end_odo - start_odo,
                                'mileage_cumulative': trip_data['Mileage_diff'].sum(),
                                'start_date': info_data.get('Installation date', ''),
                                'end_date': info_data.get('Removal date', ''),
                                'duration' : '',
                                'time': trip_data['Time_diff'].sum(),
                        
                                'speed_avg': trip_data['Mileage_diff'].sum() / trip_data['Time_diff'].sum(),
                                'fuel_consumption' : trip_data['Fuel_consumption_diff'].sum(),
                                'fuel_consumption_avg' : trip_data['Fuel_consumption_diff'].sum()*100 / trip_data['Mileage_diff'].sum()
                                
                                
                            }

                            # print(overall_metrics_df)
                            # Append the overall summary row to trip_data
                            # trip_data = trip_data.append(overall_metrics, ignore_index=True)
                            overall_metrics_df = pd.DataFrame([overall_metrics])
                            # Append trip_data to the final data
                            final_data.append(overall_metrics_df)

    except Exception as e:
        print(f"Error processing folder {folder}: {e}")





One of the required columns ['Engine_speed', 'Trip_fuel_consumption', 'Time', 'Battery_voltage'] not found in \\fileserver3\Inventory\1- Product Development\10- IPCO data Lake\1-Data Loggers\0051_Safikha_0725-0823_(29-642)_D\One_BIGfile_Refined.csv. Skipping this file.
One of the required columns ['Engine_speed', 'Trip_fuel_consumption', 'Time', 'Battery_voltage'] not found in \\fileserver3\Inventory\1- Product Development\10- IPCO data Lake\1-Data Loggers\0055_Nejat._0709-1026_(Haima_8s_red)_H\One_BIGfile_Refined.csv. Skipping this file.
One of the required columns ['Engine_speed', 'Trip_fuel_consumption', 'Time', 'Battery_voltage'] not found in \\fileserver3\Inventory\1- Product Development\10- IPCO data Lake\1-Data Loggers\0056_Nejat._0709-1026_(Haima_8s_black)_H\One_BIGfile_Refined.csv. Skipping this file.
One of the required columns ['Engine_speed', 'Trip_fuel_consumption', 'Time', 'Battery_voltage'] not found in \\fileserver3\Inventory\1- Product Development\10- IPCO data Lake\1-

In [7]:
combined_df = pd.concat(final_data, ignore_index=True)
combined_df



,plate_number,range,driver_name,vehicle_model,vehicle_option,engine-type,transmission-type,start_odo,end_odo,mileage_odo,mileage_cumulative,start_date,end_date,duration,time,speed_avg,fuel_consumption,fuel_consumption_avg
0,26-513,0001,یاسر اصالت,Dena,TC5_P_MT6,,,,,0,4060.335938,1402/02/12,1402/03/07,,178.484117,22.749004,351.244641,8.650630
1,93-953,0002,فتاحی,Peugeot_207,AT,,,,,0,5522.234375,1402/02/09,1402/03/07,,151.637040,36.417450,374.208368,6.776394
2,78-773,0003,نجات,Peugeot_207,AT,,,,,0,1964.550787,1402/02/05,1402/03/07,,62.072066,31.649515,139.184636,7.084807
3,78-773,0005,نجات,Peugeot_207,AT,,,,,0,795.242194,1402/03/07,1402/03/28,,26.475724,30.036655,60.616406,7.622383
4,93-953,0006,فتاحی,Peugeot_207,AT,,,,,0,279.125010,1402/03/07,1402/03/28,,8.097185,34.471857,22.512258,8.065296
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,37-638,0133,نعیمایی,Dena,TC_P_MT_6,,,65154,66936,1782,1581.695312,1403/07/11,1403/08/05,,32.372662,48.858982,113.531193,7.177817
85,28-333,0134,Fatahi,Dena,TC_P_AT6,,,19263,20586,1323,1363.519533,1403/07/11,1403/08/05,,50.075682,27.229175,127.785619,9.371748
86,93-953,0135,Nejat,207i,TU5_P_AT,,,38591,40255,1664,1531.656241,1403/07/11,1403/08/05,,53.291160,28.741282,115.411482,7.535077
87,93-955,0136,Deng,Dena,TC_P_MT_6,,,45046,,-45046,1043.035163,1403/04/23,1403/08/05,,31.689652,32.914062,101.205856,9.703015


In [4]:
combined_df = pd.concat(final_data, ignore_index=True)
combined_df



,plate_number,range,driver_name,vehicle_model,vehicle_option,engine-type,transmission-type,start_odo,end_odo,mileage_odo,mileage_cumulative,start_date,end_date,duration,time,speed_avg,fuel_consumption,fuel_consumption_avg
0,26-513,0001,یاسر اصالت,Dena,TC5_P_MT6,,,,,0,4060.335938,1402/02/12,1402/03/07,,178.484117,22.749004,351.244641,8.650630
1,93-953,0002,فتاحی,Peugeot_207,AT,,,,,0,5522.234375,1402/02/09,1402/03/07,,151.637040,36.417450,374.208368,6.776394
2,78-773,0003,نجات,Peugeot_207,AT,,,,,0,1964.550787,1402/02/05,1402/03/07,,62.072066,31.649515,139.184636,7.084807
3,78-773,0005,نجات,Peugeot_207,AT,,,,,0,-912862.984368,1402/03/07,1402/03/28,,26.475724,-34479.245161,60.616406,-0.006640
4,93-953,0006,فتاحی,Peugeot_207,AT,,,,,0,279.125010,1402/03/07,1402/03/28,,8.097185,34.471857,22.512258,8.065296
5,26-513,0007,یاسر اصالت,Dena,MT6_TC5_P,,,,,0,7978.628906,1402/03/07,1402/03/28,,252.739596,31.568575,723.348701,9.066078
6,18-216,0009,پیرمحمدی,Dena,AT4_TC_P,,,,,0,2953.339844,1401/10/19,1402/04/05,,142.673501,20.699989,242.826527,8.222099


In [8]:

output_file = r'C:\Users\s_alizadehnia\Desktop\output3.xlsx'

with pd.ExcelWriter(output_file) as writer:
    combined_df.to_excel(writer, sheet_name='Sheet1', index=False)

In [ ]:
combined_df = pd.concat(final_data, ignore_index=True)
combined_df

In [ ]:
combined_df = pd.concat(final_data, ignore_index=True)
combined_df

In [ ]:
trip_data['Fuel_consumption_diff']

In [ ]:
3600000

In [ ]:
*10-4

In [ ]:
# Combine all data into a single DataFrame
if final_data:
    combined_df = pd.concat(final_data, ignore_index=True)

    # Save to an Excel file
    with pd.ExcelWriter(output_file) as writer:
        combined_df.to_excel(writer, sheet_name='Sheet1', index=False)

    print(f"Data saved to {output_file}")
else:
    print("No data found.")
